# pocketHb 06 — iPhone inference + per-user calibration

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jayanthvee/pocketHb/blob/main/notebooks/06_iphone_inference.ipynb)

end-to-end pipeline on your own iphone captures. reads photos from `user_data/`, runs the global model from HF Hub (`bubbaonbubba/pockethb-base`), then fits the affine personalization layer against your real bloodwork.

before running: follow `docs/capture_protocol.md` to take ~15 photos and drop them into `user_data/`. then set the two constants in cell 2.

In [ ]:
import os, sys, subprocess
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir('..')

if not Path('scripts/download_data.py').exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/jayanthvee/pocketHb.git'])
    os.chdir('pocketHb')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])

sys.path.insert(0, 'src')
print('cwd:', Path.cwd())

In [ ]:
# ============================================================
# USER CONFIG — edit these two before running
# ============================================================
USER_HB_G_PER_DL = 15.3        # your most recent CBC value in g/dL
USER_PHOTO_DIR = 'user_data'   # folder containing your iphone .jpg captures
# ============================================================

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pockethb.inference import InferenceSession

photo_dir = Path(USER_PHOTO_DIR)
photos = sorted([p for p in photo_dir.glob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.heic'}])

if not photos:
    raise SystemExit(
        f'no images found in {photo_dir.resolve()}.\n'
        f'follow docs/capture_protocol.md, drop ~15 photos into that folder, and re-run.'
    )
print(f'found {len(photos)} photos in {photo_dir.resolve()}')
for p in photos[:5]:
    print(f'  {p.name}')
if len(photos) > 5:
    print(f'  ... ({len(photos) - 5} more)')

In [ ]:
# quick visual sanity: show the first 4 photos to confirm they're the right thing
n_show = min(4, len(photos))
fig, axes = plt.subplots(1, n_show, figsize=(4 * n_show, 4))
if n_show == 1:
    axes = [axes]
for ax, p in zip(axes, photos[:n_show]):
    ax.imshow(Image.open(p))
    ax.set_title(p.name, fontsize=9)
    ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# load the global model from HF Hub (one-time download, then cached)
sess = InferenceSession.from_hub()
print(f'backbone: {sess.backbone_name}')
print(f'global model trained on n={sess.bundle["n_train_patients"]} patients (nail only)')

## step 1 — global model raw predictions

In [ ]:
raw_per = sess.predict_per_image(photos)
raw_agg = float(np.mean(raw_per))
raw_bias = raw_agg - USER_HB_G_PER_DL

print(f'per-photo global Hb estimates (g/dL):')
for p, v in zip(photos, raw_per):
    print(f'  {p.name:30s}  {v:6.2f}')
print()
print(f'global session mean:  {raw_agg:6.2f} g/dL')
print(f'your true Hb:         {USER_HB_G_PER_DL:6.2f} g/dL')
print(f'systematic bias:      {raw_bias:+.2f} g/dL')

## step 2 — fit per-user affine calibration

In [ ]:
cal = sess.calibrate(photos, USER_HB_G_PER_DL)
personal_per = cal.predict(raw_per)
personal_agg = float(np.mean(personal_per))

print(f'calibrator: mode={cal.mode}  a={cal.a:.4f}  b={cal.b:+.4f}  n_anchors={cal.n_anchors_used}')
print()
print(f'per-photo PERSONALISED Hb estimates (g/dL):')
for p, raw, personal in zip(photos, raw_per, personal_per):
    print(f'  {p.name:30s}  raw={raw:6.2f}  personal={personal:6.2f}  err={personal-USER_HB_G_PER_DL:+.2f}')
print()
print(f'personal session mean:  {personal_agg:6.2f} g/dL (truth: {USER_HB_G_PER_DL})')
print(f'per-photo MAE pre-cal:  {np.mean(np.abs(raw_per - USER_HB_G_PER_DL)):.3f} g/dL')
print(f'per-photo MAE post-cal: {np.mean(np.abs(personal_per - USER_HB_G_PER_DL)):.3f} g/dL')

## step 3 — visualise

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
idx = np.arange(len(photos))
ax.axhline(USER_HB_G_PER_DL, color='black', ls='--', lw=1.2, label=f'your truth = {USER_HB_G_PER_DL}')
ax.scatter(idx, raw_per, color='red', s=70, alpha=0.7, label=f'global raw  MAE={np.mean(np.abs(raw_per-USER_HB_G_PER_DL)):.2f}')
ax.scatter(idx, personal_per, color='green', s=70, alpha=0.7, label=f'personalised MAE={np.mean(np.abs(personal_per-USER_HB_G_PER_DL)):.2f}')
for i in idx:
    ax.plot([i, i], [raw_per[i], personal_per[i]], color='grey', lw=0.5, alpha=0.5)
ax.set_xticks(idx)
ax.set_xticklabels([p.stem for p in photos], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Hb estimate (g/dL)')
ax.set_title('per-photo: global vs personalised on your iPhone captures')
ax.legend(loc='best'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## takeaways

- the **systematic bias** number above (raw_agg − truth) is what the per-user calibrator subtracts. once subtracted, predictions cluster around your real Hb instead of around the global model's biased estimate.
- the **residual per-photo MAE post-cal** is the irreducible noise: photo-to-photo variation that no scalar correction can remove. averaging over multiple photos at inference time reduces it.
- this is the personalization result that goes into the medium piece and the twitter demo (chunks 10–12). it's also the live-demo loop on the HF Space (chunk 8).
- **still not a medical device.** the personalized number is sharper than the global one for YOU, but tells you nothing about another user without re-calibrating to their own bloodwork.